 # RLVR Part 3 Analysis

 Notebook converted to percent format with text-based data outputs for preservation.

In [1]:
# results_save_filename = "oos_results_ent_0.001_pen_1.0_lr_0.0003.pkl"
# results_save_filename = "oos_results_ent_0.01_pen_1.0_lr_0.0003.pkl"
# results_save_filename = "oos_results_seed_987654_ent_0.01_pen_1.0_lr_0.0003.pkl"
results_save_filename = (
    "oos_results_seed_987654_update4_ent_0.001_pen_1.0_lr_0.0003.pkl"
)

 ### CELL 1: Setup & Load the OOS Data

In [2]:
import pickle
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from plotly.subplots import make_subplots
from IPython.display import display

# Set your local PC path
rl_root = Path(r"C:\Users\ping\Files_win10\python\py311\stocks\notebooks_RLVR_v2")
output_dir = rl_root / "output"

pkl_path = output_dir / results_save_filename

if not pkl_path.exists():
    raise FileNotFoundError(f"Cannot find {pkl_path}. Ensure local directory synced!")

# Load the OOS Results
with open(pkl_path, "rb") as f:
    results = pickle.load(f)

# Convert the blotter (list of dicts) into a Pandas DataFrame
blotter_df = pd.DataFrame(results["blotter"])

# Format the date column properly using 'decision_date'
blotter_df["date"] = pd.to_datetime(blotter_df["decision_date"])
blotter_df = blotter_df.sort_values("date").reset_index(drop=True)

print(f"✅ Data successfully loaded. Total days evaluated: {len(blotter_df)}")

✅ Data successfully loaded. Total days evaluated: 1073


In [3]:
print(blotter_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1073 entries, 0 to 1072
Data columns (total 25 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   decision_date       1073 non-null   object        
 1   buy_date            1073 non-null   object        
 2   sell_date           1073 non-null   object        
 3   universe_size       1073 non-null   int64         
 4   max_score           1073 non-null   float64       
 5   min_score           1073 non-null   float64       
 6   observation         1073 non-null   object        
 7   raw_actions         1073 non-null   object        
 8   decoded_offset      1073 non-null   int64         
 9   decoded_width       1073 non-null   int64         
 10  top_3_tickers       1073 non-null   object        
 11  chosen_tickers      1073 non-null   object        
 12  predicted_reward    1073 non-null   float64       
 13  rl_reward_received  1073 non-null   float64     

 ### CELL 2: Champion Model & High-Level OOS Metrics

In [4]:
metadata = results.get("metadata", {})
champion_file = metadata.get("CHAMPION_FILE", "Unknown (Check metadata logging)")
best_seed = metadata.get("BEST_SEED", "Unknown")

print(f"{'='*50}")
print(f"🏆 CHAMPION MODEL LOADED: {champion_file}")
print(f"   results_save_filename: {results_save_filename}")
print(f"🌱 Winning Seed: {best_seed}")
print(f"{'='*50}")

print(f"\n📈 Out-Of-Sample Performance:")
print(f"Total Cumulative Return : {results['total_return']*100:.2f}%")
print(f"Sharpe Ratio (Ann)      : {results['sharpe_ratio']:.3f}")
print(f"Total Trading Steps     : {results['steps']} days")

🏆 CHAMPION MODEL LOADED: model_seed_987654_update4_ent_0.001_pen_1.0_lr_0.0003_seed_987654_ep_51_sh_1.66.pt
   results_save_filename: oos_results_seed_987654_update4_ent_0.001_pen_1.0_lr_0.0003.pkl
🌱 Winning Seed: 987654

📈 Out-Of-Sample Performance:
Total Cumulative Return : 52.28%
Sharpe Ratio (Ann)      : 1.004
Total Trading Steps     : 1073 days


### Visualizing the Agent's Brain (Training Diagnostics)
This shows how the Neural Network converged over time.
* **Policy Loss**: How much the agent is changing its strategy (should stabilize).
* **Value Loss**: How accurate the Critic is at predicting rewards (should drop).
* **Entropy**: The "Curiosity" dial (gradually drops as the agent becomes confident).
* **Avg Reward**: The raw alpha the agent achieved in the training environment.

In [5]:
training_history = results.get("training_history", [])
metadata = results.get("metadata", {})

if training_history:
    history_df = pd.DataFrame(training_history)

    # Let's print out the exact parameters used to train this model
    print("=" * 50)
    print("⚙️ HYPERPARAMETERS USED FOR THIS RUN:")
    for k, v in metadata.items():
        print(f" - {k.ljust(20)} : {v}")
    print("=" * 50)

    # Plotting
    fig = make_subplots(
        rows=2,
        cols=2,
        subplot_titles=(
            "Total & Policy Loss",
            "Value Loss (Prediction Error)",
            "Entropy (Curiosity/Exploration)",
            "Average Step Reward",
        ),
        shared_xaxes=True,
        vertical_spacing=0.1,
    )

    # Top Left: Total Loss & Policy Loss
    fig.add_trace(
        go.Scatter(
            x=history_df["epoch"],
            y=history_df["total_loss"],
            mode="lines",
            name="Total Loss",
        ),
        row=1,
        col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=history_df["epoch"],
            y=history_df["policy_loss"],
            mode="lines",
            name="Policy Loss",
        ),
        row=1,
        col=1,
    )

    # Top Right: Value Loss (The Critic)
    fig.add_trace(
        go.Scatter(
            x=history_df["epoch"],
            y=history_df["value_loss"],
            mode="lines",
            name="Value Loss",
            line=dict(color="orange"),
        ),
        row=1,
        col=2,
    )

    # Bottom Left: Entropy
    fig.add_trace(
        go.Scatter(
            x=history_df["epoch"],
            y=history_df["entropy"],
            mode="lines",
            name="Entropy",
            line=dict(color="green"),
        ),
        row=2,
        col=1,
    )

    # Bottom Right: Avg Epoch Reward
    fig.add_trace(
        go.Scatter(
            x=history_df["epoch"],
            y=history_df["avg_reward"],
            mode="lines",
            name="Avg Reward",
            line=dict(color="purple"),
        ),
        row=2,
        col=2,
    )

    fig.update_layout(
        height=700,
        width=1000,
        title_text="🧠 Agent Training Diagnostics (Winning Seed)",
    )
    fig.show()
else:
    print(
        "⚠️ No training history found in this pickle file. (Was it trained using the old script?)"
    )

⚙️ HYPERPARAMETERS USED FOR THIS RUN:
 - ENT_COEF             : 0.001
 - EVAL_FREQ            : 3
 - LR                   : 0.0003
 - MINI_BATCH_SIZE      : 1024
 - NUM_ENVS             : 16
 - NUM_EPOCHS           : 75
 - NUM_STEPS            : 256
 - PATIENCE             : 15
 - SEEDS                : [987654]
 - UPDATE_EPOCHS        : 4
 - WARMUP_EPOCHS        : 5
 - downside_penalty     : 1.0
 - BEST_SEED            : 987654
 - CHAMPION_FILE        : model_seed_987654_update4_ent_0.001_pen_1.0_lr_0.0003_seed_987654_ep_51_sh_1.66.pt


 ### CELL 3: Parsing the Action Dimensions (The Agent's Personality)

In [6]:
import sys
from pathlib import Path

# Ensure the root folder is in path so we can import core modules
if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))

from core.settings import TradingConfig
from strategy.registry import get_strategy_registry

# 1. Dynamically fetch strategies from the live system
config = TradingConfig()
registry = get_strategy_registry(config)

# 2. Build Action Names
# The first N actions are the feature weights
action_names = list(registry.keys())

# The last 2 actions are ALWAYS the portfolio bounding parameters
# UPDATED: Renamed from 'Rank Offset (Start Pos)' to reflect the new architecture
action_names.extend(["Rank Offset Percentile", "Rank Width"])

# 3. Extract raw actions
actions = np.vstack(blotter_df["raw_actions"].values)

# 4. Map to DataFrame
import sys
from pathlib import Path

# Ensure the root folder is in path so we can import core modules
if str(rl_root) not in sys.path:
    sys.path.append(str(rl_root))

from core.settings import TradingConfig
from strategy.registry import get_strategy_registry

# 1. Dynamically fetch strategies from the live system
config = TradingConfig()
registry = get_strategy_registry(config)

# 2. Build Action Names
# The first N actions are the feature weights
action_names = list(registry.keys())

# The last 2 actions are ALWAYS the portfolio bounding parameters
# UPDATED: Renamed from 'Rank Offset (Start Pos)' to reflect the new architecture
action_names.extend(["Rank Offset Percentile", "Rank Width"])

# 3. Extract raw actions
actions = np.vstack(blotter_df["raw_actions"].values)

# 4. Map to DataFrame
print(f"action_names")
for i, name in enumerate(action_names):
    if i < actions.shape[1]:
        blotter_df[name] = actions[:, i]
    print(f"{i:02d}    {name}")

print(
    f"\n✅ Agent actions unpacked. Detected {len(registry)} strategies + 2 portfolio bounds ({len(action_names)} total dims)."
)

action_names
00    Log Price Gain
01    Sharpe (TRP)
02    Momentum (21d)
03    Info Ratio (63d)
04    Oversold (-RSI)
05    Dip Buyer (-dd_21)
06    Range Position (20d)
07    Return Autocorr (15d)
08    Low Volatility (-ATRP)
09    Slope_P_5_Z
10    Slope_V_5_Z
11    Convexity
12    Rank Offset Percentile
13    Rank Width

✅ Agent actions unpacked. Detected 12 strategies + 2 portfolio bounds (14 total dims).


 ### CELL 4: Visualizing the Agent's "Personality" (Action Distribution)

In [7]:
# Calculate the mean weight for each strategy across the entire OOS period
mean_actions = blotter_df[action_names].mean().reset_index()
mean_actions.columns = ["Strategy", "Average Weight"]
mean_actions = mean_actions.sort_values("Average Weight", ascending=True)

fig = px.bar(
    mean_actions,
    x="Average Weight",
    y="Strategy",
    orientation="h",
    title="🤖 Agent 'Personality' - Average Strategy Weights (OOS)",
    color="Average Weight",
    color_continuous_scale="Viridis",
)
fig.update_layout(height=600, width=900)
fig.show()

 ### CELL 5: Equity Curve, Alpha Curve, & Drawdown Chart vs Benchmark

In [8]:
hp = results.get("metadata", {}).get("HOLDING_PERIOD", 5)
blotter_df["Mkt_Impact"] = blotter_df["mkt_return"] / hp
blotter_df["Benchmark_Equity"] = (1 + blotter_df["Mkt_Impact"]).cumprod()

# Calculate Agent Drawdown
blotter_df["Peak"] = blotter_df["agent_equity"].cummax()
blotter_df["Drawdown"] = (blotter_df["agent_equity"] / blotter_df["Peak"]) - 1.0

# Create Subplots (3 Rows)
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.5, 0.25, 0.25],
    subplot_titles=(
        "📈 OOS Equity Curve (Agent vs SPY)",
        "🔥 Cumulative Alpha Equity (Outperformance)",
        "📉 Agent Drawdown Profile",
    ),
)

# Top Pane: Agent Equity
fig.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=blotter_df["agent_equity"],
        mode="lines",
        name="Agent Strategy",
        line=dict(color="blue", width=2),
    ),
    row=1,
    col=1,
)

# Top Pane: Benchmark Equity
fig.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=blotter_df["Benchmark_Equity"],
        mode="lines",
        name="Benchmark (SPY)",
        line=dict(color="gray", width=2, dash="dash"),
    ),
    row=1,
    col=1,
)

# Middle Pane: Alpha Equity
fig.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=blotter_df["alpha_equity"],
        mode="lines",
        name="Alpha Equity",
        line=dict(color="purple", width=2),
    ),
    row=2,
    col=1,
)
fig.add_hline(y=1.0, line_dash="dot", line_color="black", row=2, col=1)

# Bottom Pane: Drawdown Area
fig.add_trace(
    go.Scatter(
        x=blotter_df["date"],
        y=blotter_df["Drawdown"],
        mode="lines",
        fill="tozeroy",
        name="Drawdown",
        line=dict(color="red", width=1),
    ),
    row=3,
    col=1,
)

fig.update_layout(
    height=900,
    width=1000,
    hovermode="x unified",
    title_text="Performance & Risk Profiling (Slippage Applied)",
)
fig.update_yaxes(title_text="Multiplier", row=1, col=1)
fig.update_yaxes(title_text="Alpha Multiplier", row=2, col=1)
fig.update_yaxes(title_text="Drawdown %", tickformat=".1%", row=3, col=1)
fig.show()

 ### CELL 6: DataFrame Info Check

In [9]:
blotter_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1073 entries, 0 to 1072
Data columns (total 43 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   decision_date           1073 non-null   object        
 1   buy_date                1073 non-null   object        
 2   sell_date               1073 non-null   object        
 3   universe_size           1073 non-null   int64         
 4   max_score               1073 non-null   float64       
 5   min_score               1073 non-null   float64       
 6   observation             1073 non-null   object        
 7   raw_actions             1073 non-null   object        
 8   decoded_offset          1073 non-null   int64         
 9   decoded_width           1073 non-null   int64         
 10  top_3_tickers           1073 non-null   object        
 11  chosen_tickers          1073 non-null   object        
 12  predicted_reward        1073 non-null   float64 

In [10]:
# 1. Get the value of the last row
last_value = blotter_df["raw_actions"].iloc[-1]

# 2. Get the length of that value (assuming it's a string, list, etc.)
last_value_len = len(last_value)

print(f"Value: {last_value}")
print(f"Length: {last_value_len}")

Value: [0.061945170164108276, 0.06906019151210785, -0.14943347871303558, 0.30311325192451477, 0.12967665493488312, 0.15013937652111053, -0.06566928327083588, 0.07074497640132904, -0.034246716648340225, -0.06183743476867676, -0.3246705234050751, -0.1586022526025772, -0.5217182636260986, 0.4474368691444397]
Length: 14


 ### CELL 7: Sample Selected Tickers

In [11]:
# Print chosen tickers along with their corresponding dates (first and last rows)
first_idx = 0
last_idx = -1

first_date = blotter_df["date"].iloc[first_idx]
last_date = blotter_df["date"].iloc[last_idx]

print(f"Sample chosen tickers from the first row ({first_date.strftime('%Y-%m-%d')}):")
print(blotter_df["chosen_tickers"].iloc[first_idx])

print(f"\nSample chosen tickers from the last row ({last_date.strftime('%Y-%m-%d')}):")
print(blotter_df["chosen_tickers"].iloc[last_idx])

Sample chosen tickers from the first row (2022-04-01):
['KDP', 'DKNG', 'ITOT', 'DIA', 'SPYM']

Sample chosen tickers from the last row (2026-07-14):
['TECL', 'SITM', 'TYL', 'TWLO', 'ACGL', 'TEM', 'CNQ']


 ### CELL 8: Pandas Display Settings

In [12]:
# Force Pandas to display all columns without truncating in text representation
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 2000)

 ### CELL 9: Rank Start, Offset, and Universe Tickers

In [13]:
sample_idx = 0
row = blotter_df.iloc[sample_idx]

u_size = row.get("universe_size", "N/A")
r_offset = row.get("decoded_offset", "N/A")
r_width = row.get("decoded_width", "N/A")
selected = row.get("chosen_tickers", [])
num_selected = len(selected)

print(f"=== Universe Selection on {row['date'].strftime('%Y-%m-%d')} ===")
print(f"Total Available Universe (universe_size) : {u_size}")
print(f"Calculated Rank Start (offset)           : {r_offset}")
print(f"Calculated Rank Width (width)            : {r_width}")
print(f"Final Tickers Bought                     : {num_selected}")
print(f"Sample Tickers Bought: {selected[:10]} ...")
print(f"blotter row[{sample_idx}]:\n{row}")

=== Universe Selection on 2022-04-01 ===
Total Available Universe (universe_size) : 878
Calculated Rank Start (offset)           : 379
Calculated Rank Width (width)            : 5
Final Tickers Bought                     : 5
Sample Tickers Bought: ['KDP', 'DKNG', 'ITOT', 'DIA', 'SPYM'] ...
blotter row[0]:
decision_date                                                    2022-04-01
buy_date                                                         2022-04-04
sell_date                                                        2022-04-11
universe_size                                                           878
max_score                                                          2.559233
min_score                                                         -1.530883
observation               [-0.14340397715568542, -0.3984471559524536, 0....
raw_actions               [0.2933565378189087, 0.16115956008434296, -0.1...
decoded_offset                                                          379
decoded_w

 ### CELL 10: Top 5 Best Days & Crash Analysis

In [14]:
print("🌟 TOP 5 HIGHEST RETURN DAYS (Net of Slippage) 🌟")

top_5 = blotter_df.nlargest(5, "actual_return")[
    ["date", "actual_return", "alpha", "agent_equity"] + action_names[:3]
].copy()

# Format numeric columns for output readability
top_5["date"] = top_5["date"].dt.strftime("%Y-%m-%d")
top_5["actual_return"] = top_5["actual_return"].apply(lambda x: f"{x:.4%}")
top_5["alpha"] = top_5["alpha"].apply(lambda x: f"{x:.4%}")
top_5["agent_equity"] = top_5["agent_equity"].apply(lambda x: f"{x:.3f}")
for col in action_names[:3]:
    top_5[col] = top_5[col].apply(lambda x: f"{x:.4f}")

# Output as a clean text table
print(top_5.to_string(index=False))

print("\n" + "=" * 50)
print("🚨 CRASH ANALYSIS (Worst Single-Day Return) 🚨")
worst_idx = blotter_df["actual_return"].idxmin()
worst_day = blotter_df.iloc[worst_idx]
day_before = blotter_df.iloc[max(0, worst_idx - 1)]

print(
    f"Worst Day: {worst_day['date'].strftime('%Y-%m-%d')} | "
    f"Return: {worst_day['actual_return']*100:.2f}% | "
    f"Portfolio Value: {worst_day['agent_equity']:.3f}"
)
print("-" * 50)
print("🔍 What was the agent's strategy allocation going into this crash?")

crash_weights = pd.DataFrame(
    {
        "Strategy": action_names,
        "Weight on Worst Day": worst_day[action_names].values,
        "Weight Day Prior": day_before[action_names].values,
    }
).copy()

# Formatting weights for display
crash_weights_formatted = crash_weights.copy()
crash_weights_formatted["Weight on Worst Day"] = crash_weights_formatted[
    "Weight on Worst Day"
].apply(lambda x: f"{x:.6f}")
crash_weights_formatted["Weight Day Prior"] = crash_weights_formatted[
    "Weight Day Prior"
].apply(lambda x: f"{x:.6f}")

print("\nStrategy Allocation Details (Pre-and-Post Crash):")
print(crash_weights_formatted.to_string(index=False))

fig = px.line_polar(
    crash_weights,
    r="Weight on Worst Day",
    theta="Strategy",
    line_close=True,
    title=f"Radar: Agent's Brain on {worst_day['date'].strftime('%Y-%m-%d')} (Crash Day)",
)
fig.update_traces(fill="toself", line_color="red")
fig.show()

🌟 TOP 5 HIGHEST RETURN DAYS (Net of Slippage) 🌟
      date actual_return    alpha agent_equity Log Price Gain Sharpe (TRP) Momentum (21d)
2025-04-28      17.2669% 16.4587%        1.319         0.0314      -0.0891        -0.0920
2026-04-09      15.7232% 11.2079%        1.442         0.1722      -0.0253        -0.4182
2025-05-06      14.6437%  9.9320%        1.386         0.0256      -0.1018         0.2037
2025-04-17      14.2705%  7.0762%        1.262        -0.0270       0.0130         0.1410
2023-11-24      13.9241% 13.4380%        1.009        -0.2265      -0.0333        -0.1791

🚨 CRASH ANALYSIS (Worst Single-Day Return) 🚨
Worst Day: 2025-05-15 | Return: -15.70% | Portfolio Value: 1.351
--------------------------------------------------
🔍 What was the agent's strategy allocation going into this crash?

Strategy Allocation Details (Pre-and-Post Crash):
              Strategy Weight on Worst Day Weight Day Prior
        Log Price Gain            0.140876         0.311686
          Sha

 ### CELL 11: Observation Space Sanity Check (Hunting for Extremes)

In [15]:
print("🔍 EXTRACTING OBSERVATION SPACE WITH DYNAMIC LABELS...")

# 1. Extract raw observation arrays
obs_raw = blotter_df["observation"].values
obs_array = np.vstack([np.array(x).flatten() for x in obs_raw])
num_dims = obs_array.shape[1]

# 2. Dynamically build the Cross-Sectional Features matching Phase 3 logic
from strategy.registry import get_strategy_registry
from core.settings import TradingConfig

registry = get_strategy_registry(TradingConfig())
cross_sectional_features = []

for name, blueprint in registry.items():
    # Matches the exact logic in RLOracle.compute_alpha_matrix
    suffix = " (Z)" if blueprint.scaling_type == "Z-Score" else " (S)"
    cross_sectional_features.append(f"{name}{suffix}")

# Macro features remain hardcoded because they come from data_pipeline, not registry
macro_features = [
    "Mkt_Ret",
    "Mkt_Ret_Z",
    "Macro_Trend",
    "Macro_Trend_Z",
    "High_Yield_Spread_Z",
    "Yield_Curve_10Y2Y_Z",
    "Macro_Trend_Vel_Z",
    "Macro_Trend_Mom",
    "Macro_Vix_Z",
    "Macro_Vix_Ratio",
    "Mkt_Vol_63d_Z",
]

# Generate the full list of mapped column names (Mean, then Std, then Macro)
mapped_names = (
    [f"Mean: {f}" for f in cross_sectional_features]
    + [f"Std: {f}" for f in cross_sectional_features]
    + macro_features
)


# 3. Create 'obs_df' globally with descriptive column headers & dates
obs_df = pd.DataFrame(obs_array, columns=mapped_names, index=blotter_df["date"])

# 4. Compute summary statistics
desc_stats = pd.DataFrame(obs_array).describe().T[["mean", "std", "min", "max"]]

# 5. Assemble mapping table
mapping_table = pd.DataFrame(
    {
        "Dim Index": [f"Dim_{i:02d}" for i in range(num_dims)],
        "Feature Name": mapped_names,
        "Mean": desc_stats["mean"].values,
        "Std Dev": desc_stats["std"].values,
        "Min": desc_stats["min"].values,
        "Max": desc_stats["max"].values,
    }
)


# Formatting helper to append [CLIP] to values near the +/- 4.0 threshold
def format_with_clip(val):
    if pd.isna(val):
        return ""
    formatted = f"{val:.5f}"
    if abs(abs(val) - 4.0) < 1e-5:
        return f"{formatted} [CLIP]"
    return formatted


# Apply formatting for console print output
mapping_table_print = mapping_table.copy()
mapping_table_print["Mean"] = mapping_table_print["Mean"].apply(lambda x: f"{x:.5f}")
mapping_table_print["Std Dev"] = mapping_table_print["Std Dev"].apply(
    lambda x: f"{x:.5f}"
)
mapping_table_print["Min"] = mapping_table_print["Min"].apply(format_with_clip)
mapping_table_print["Max"] = mapping_table_print["Max"].apply(format_with_clip)

print("\n--- Mapped Observation Space Statistics ---")
print(mapping_table_print.to_string(index=False))

print("\n📈 GENERATING BOX PLOT OF OBSERVATION DIMENSIONS...")
fig = go.Figure()

for col in obs_df.columns:
    fig.add_trace(
        go.Box(
            y=obs_df[col],
            name=col,
            boxpoints="outliers",
            jitter=0.5,
            marker=dict(size=2),
        )
    )

fig.update_layout(
    title="🤖 Agent Observation Space Distribution (Check for wild outliers)",
    yaxis_title="Value",
    xaxis_title="Observation Dimension",
    height=600,
    width=1000,
    showlegend=False,
    shapes=[
        dict(
            type="line",
            y0=3,
            y1=3,
            x0=-0.5,
            x1=num_dims - 0.5,
            line=dict(color="orange", width=1, dash="dash"),
        ),
        dict(
            type="line",
            y0=-3,
            y1=-3,
            x0=-0.5,
            x1=num_dims - 0.5,
            line=dict(color="orange", width=1, dash="dash"),
        ),
        dict(
            type="line",
            y0=4.0,
            y1=4.0,
            x0=-0.5,
            x1=num_dims - 0.5,
            line=dict(color="red", width=1, dash="dash"),
        ),
        dict(
            type="line",
            y0=-4.0,
            y1=-4.0,
            x0=-0.5,
            x1=num_dims - 0.5,
            line=dict(color="red", width=1, dash="dash"),
        ),
    ],
)
fig.show()

🔍 EXTRACTING OBSERVATION SPACE WITH DYNAMIC LABELS...

--- Mapped Observation Space Statistics ---
Dim Index                     Feature Name     Mean Std Dev      Min     Max
   Dim_00         Mean: Log Price Gain (Z)  0.06034 0.60268 -1.65758 1.52184
   Dim_01           Mean: Sharpe (TRP) (S)  0.07370 0.78058 -1.88371 1.99650
   Dim_02         Mean: Momentum (21d) (S)  0.02888 0.67832 -1.68316 1.66027
   Dim_03       Mean: Info Ratio (63d) (S) -0.69894 0.80531 -2.15995 1.40984
   Dim_04        Mean: Oversold (-RSI) (S) -0.06399 0.77488 -1.70494 1.93908
   Dim_05     Mean: Dip Buyer (-dd_21) (S) -0.10936 0.57441 -0.95658 1.72525
   Dim_06   Mean: Range Position (20d) (S)  0.00053 0.80559 -1.80368 1.41922
   Dim_07  Mean: Return Autocorr (15d) (S)  0.03653 0.72262 -2.12860 2.01885
   Dim_08 Mean: Low Volatility (-ATRP) (Z)  0.22702 0.32218 -0.53391 0.62780
   Dim_09            Mean: Slope_P_5_Z (Z)  0.02052 0.74021 -2.31708 1.88733
   Dim_10            Mean: Slope_V_5_Z (Z)  0.01573 0.